# Assignment 3 – Descriptive Analytics
## Two Averages, One Argument

**Student work notebook**  
Course: ICA-F26 · The Prices Desk  
Week 3 · Descriptive Analytics

This notebook computes the mean, median and first-to-last percentage change for each staple using the cleaned teaching file.  
All calculations are reproducible from the source CSV.

In [1]:
import pandas as pd
import numpy as np

# Load the cleaned teaching data (same folder as notebook)
df = pd.read_csv("teaching_prices_clean.csv")
df['date'] = pd.to_datetime(df['date'])
print("Shape:", df.shape)
print("Date range:", df['date'].min().date(), "→", df['date'].max().date())
print("Items:", df['item'].unique().tolist())
print("Cities:", df['city'].unique().tolist())
df.head()


Shape: (96, 6)
Date range: 2026-01-01 → 2026-03-19
Items: ['Staple A', 'Oil B', 'Pulse C', 'Sweetener D']
Cities: ['City 1', 'City 2']


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS


## 1. Core statistics per item

For each item we report:
- number of price observations
- arithmetic mean
- median
- first-to-last percentage change (using the simple average of the two cities on the first and last observation dates)

In [2]:
def first_last_pct(group):
    first_date = group['date'].min()
    last_date  = group['date'].max()
    first_avg  = group.loc[group['date'] == first_date, 'price'].mean()
    last_avg   = group.loc[group['date'] == last_date,  'price'].mean()
    return ((last_avg - first_avg) / first_avg) * 100

stats = []
for item, g in df.groupby('item'):
    stats.append({
        'item': item,
        'n_obs': len(g),
        'mean': round(g['price'].mean(), 2),
        'median': round(g['price'].median(), 2),
        'first_to_last_pct': round(first_last_pct(g), 2)
    })

stats_df = pd.DataFrame(stats).set_index('item')
stats_df

,n_obs,mean,median,first_to_last_pct
item,,,,
Oil B,24,324.83,324.00,15.28
Pulse C,24,205.92,206.00,3.69
Staple A,24,111.17,111.75,12.98
Sweetener D,24,166.75,166.75,19.41


## 2. City-level view (optional detail)

City 2 is consistently a few rupees higher than City 1 for every item. The percentage changes remain very similar across cities.

In [3]:
city_stats = []
for (item, city), g in df.groupby(['item', 'city']):
    first_p = g.loc[g['date'] == g['date'].min(), 'price'].values[0]
    last_p  = g.loc[g['date'] == g['date'].max(), 'price'].values[0]
    pct = ((last_p - first_p) / first_p) * 100
    city_stats.append({
        'item': item,
        'city': city,
        'mean': round(g['price'].mean(), 2),
        'median': round(g['price'].median(), 2),
        'first': first_p,
        'last': last_p,
        'pct_change': round(pct, 2)
    })

pd.DataFrame(city_stats)

,item,city,mean,median,first,last,pct_change
0,Oil B,City 1,321.83,321.00,298.0,344.0,15.44
1,Oil B,City 2,327.83,327.00,304.0,350.0,15.13
2,Pulse C,City 1,202.92,202.50,200.0,207.5,3.75
3,Pulse C,City 2,208.92,208.50,206.0,213.5,3.64
4,Staple A,City 1,108.17,107.75,101.0,114.5,13.37
5,Staple A,City 2,114.17,113.75,107.0,120.5,12.62
6,Sweetener D,City 1,163.75,163.75,149.0,178.5,19.80
7,Sweetener D,City 2,169.75,169.75,155.0,184.5,19.03


## 3. Effect of a flagged value (sensitivity check)

Suppose one observation for **Oil B – City 1** on 2026-02-26 was flagged as suspicious (price recorded as 380 instead of the true 331).  
We recompute mean and median for Oil B with and without that flagged observation.

In [4]:
oil = df[df['item'] == 'Oil B'].copy()

# Inject a synthetic flagged value for demonstration
flag_mask = (oil['city'] == 'City 1') & (oil['date'] == '2026-02-26')
oil_flagged = oil.copy()
oil_flagged.loc[flag_mask, 'price'] = 380.0   # artificially high

print("With flagged value (n=24):")
print(f"  mean   = {oil_flagged['price'].mean():.2f}")
print(f"  median = {oil_flagged['price'].median():.2f}")

print("\nWithout flagged value (n=23):")
clean = oil_flagged.loc[~flag_mask]
print(f"  mean   = {clean['price'].mean():.2f}")
print(f"  median = {clean['price'].median():.2f}")

print("\nDifference caused by the single flagged observation:")
print(f"  Δ mean   = {oil_flagged['price'].mean() - clean['price'].mean():.2f}")
print(f"  Δ median = {oil_flagged['price'].median() - clean['price'].median():.2f}")

With flagged value (n=24):
  mean   = 326.88
  median = 324.00

Without flagged value (n=23):
  mean   = 324.57
  median = 324.00

Difference caused by the single flagged observation:
  Δ mean   = 2.31
  Δ median = 0.00


## 4. Which statistic would I report?

For a short referee note about whether prices “really went up” I would lead with the **first-to-last percentage change** (it answers the direction and magnitude question directly) and support it with the **median** as a robust central tendency measure.  
The mean is useful but more sensitive to any remaining outliers; the median and the endpoint change tell a clearer story for a non-technical audience.

In [5]:
# Final clean summary table for the referee note
summary = stats_df[['mean', 'median', 'first_to_last_pct']].copy()
summary.columns = ['Mean (PKR/kg)', 'Median (PKR/kg)', 'First-to-last % change']
summary

,Mean (PKR/kg),Median (PKR/kg),First-to-last % change
item,,,
Oil B,324.83,324.00,15.28
Pulse C,205.92,206.00,3.69
Staple A,111.17,111.75,12.98
Sweetener D,166.75,166.75,19.41
